In [20]:
import pandas as pd
import pycountry
import plotly.express as px
import plotly.io as pio
import imageio
import os

df = pd.read_csv("Data_Files/base_data104_with_lookup.csv", low_memory=False)
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df = df.dropna(subset=["Country", "Year"])

country_map = {
    "United States of America": "USA",
    "Czech Republic": "Czechia",
    "Republic of Korea": "Korea, Republic of",
    "Republic of Moldova": "Moldova",
    "Turkey": "Türkiye",
    "Venezuela": "Venezuela, Bolivarian Republic of",
    "United Kingdom, England and Wales": "United Kingdom",
    "United Kingdom, Scotland": "United Kingdom",
    "United Kingdom, Northern Ireland": "United Kingdom",
    "Hong Kong SAR": "Hong Kong",
    "Reunion": "France",
    "Guadeloupe": "France",
    "Martinique": "France",
    "Mayotte": "France",
    "French Guiana": "France",
    "Rodrigues": "Mauritius",
    "Puerto Rico": "United States",
    "Virgin Islands (USA)": "USA"}
df["Country_clean"] = df["Country"].replace(country_map)

def to_iso3(name):
    try:
        return pycountry.countries.lookup(name).alpha_3
    except:
        return None

df["ISO3"] = df["Country_clean"].apply(to_iso3)

#Einträge pro Jahr und Land zählen
counts = (df.groupby(["ISO3", "Year"]).size().reset_index(name="Entries"))
years = list(range(2010, 2023))

#Überprüfung ob jedes Land für jedes Jahr einen Eintrag hat
all_iso = counts["ISO3"].dropna().unique()
full_index = pd.MultiIndex.from_product([all_iso, years], names=["ISO3", "Year"])
counts_full = (counts.set_index(["ISO3", "Year"]).reindex(full_index, fill_value=0).reset_index())

#Anfänglich waren 0 Einträge farblich markiert, jetzt auf None setzen für bessere Visualisierung
counts_full.loc[counts_full["Entries"] == 0, "Entries"] = None

os.makedirs("gif_frames", exist_ok=True)

for year in years:
    sub = counts_full[counts_full["Year"] == year]

    fig = px.choropleth(sub,locations="ISO3", color="Entries", color_continuous_scale="Viridis", title=f"WHO Mortality – Data Availability {year}",range_color=(0, counts_full["Entries"].max()))

    fig.update_layout(width=1100, height=700)

    fname = f"gif_frames/frame_{year}.png"
    pio.write_image(fig, fname)

images = []
for year in years:
    img = imageio.imread(f"gif_frames/frame_{year}.png")
    images.append(img)
imageio.mimsave("WHO_Data_13_Years3.gif", images, duration=100)



Saved: gif_frames/frame_2010.png
Saved: gif_frames/frame_2011.png
Saved: gif_frames/frame_2012.png
Saved: gif_frames/frame_2013.png
Saved: gif_frames/frame_2014.png
Saved: gif_frames/frame_2015.png
Saved: gif_frames/frame_2016.png
Saved: gif_frames/frame_2017.png
Saved: gif_frames/frame_2018.png
Saved: gif_frames/frame_2019.png
Saved: gif_frames/frame_2020.png
Saved: gif_frames/frame_2021.png
Saved: gif_frames/frame_2022.png


/var/folders/37/084722ds0s5f0lsp5b75z_w40000gn/T/ipykernel_38758/37015510.py:151: DeprecationWarning:

Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.



✔ DONE — GIF saved as WHO_Data_13_Years.gif
